#  Superstore Sales Data Analysis
**Objective:** Perform a comprehensive Exploratory Data Analysis (EDA) on the Superstore Sales dataset, including data profiling, quality checks, cleaning, and visual exploration.

## 1 - Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

print(" Libraries loaded successfully")

## 2 - Load the Dataset

In [ ]:
data = pd.read_csv('dataset.csv')
data.head()

### 2.1 - Quick Look - First & Last Rows

In [ ]:
data.head(10)

In [ ]:
data.tail(10)

### 2.2 - Dataset Shape

In [ ]:
print(f"Rows   : {data.shape[0]}")
print(f"Columns: {data.shape[1]}")

---
## 3 - Data Types Overview

In [ ]:
data.info()

In [ ]:
data.dtypes

### 3.1 - Fix Date Columns to datetime
`Order Date` and `Ship Date` are read as objects - convert them to proper datetime format.

In [ ]:
data['Order Date'] = pd.to_datetime(data['Order Date'], dayfirst=True)
data['Ship Date']  = pd.to_datetime(data['Ship Date'],  dayfirst=True)
print(" Date columns converted to datetime")
data[['Order Date', 'Ship Date']].dtypes

### 3.2 - Convert Postal Code to String
Postal codes are identifiers, not numbers. Converting to string prevents unwanted arithmetic.

In [ ]:
data['Postal Code'] = data['Postal Code'].astype(str)
print(" Postal Code converted to string")
data.dtypes

### 3.3 - Feature Engineering - Shipping Days & Order Month

In [ ]:
data['Days to Ship'] = (data['Ship Date'] - data['Order Date']).dt.days
data['Order Month']  = data['Order Date'].dt.month_name()
data[['Order Date', 'Ship Date', 'Days to Ship', 'Order Month']].head()

---
## 4 - Descriptive Statistics

### 4.1 - Numerical Columns

In [ ]:
data.describe()

### 4.2 - Categorical Columns

In [ ]:
data.describe(include='object')

### 4.3 - Full Summary (All Columns)

In [ ]:
data.describe(include='all').T

---
## 5 - Data Quality Checks 

### 5.1 - Missing Values (Nulls)

In [ ]:
null_summary = data.isnull().sum().to_frame('missing_count')
null_summary['missing_pct'] = (null_summary['missing_count'] / len(data) * 100).round(2)
null_summary[null_summary['missing_count'] > 0]

In [ ]:
# Visualize missing values
plt.figure(figsize=(12, 4))
sns.heatmap(data.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.tight_layout()
plt.show()

### 5.1.1 - Handle Missing Postal Codes
The missing Postal Codes are all from Burlington, Vermont. We fill them with the correct code **05401**.

In [ ]:
missing_postal = data[data['Postal Code'] == 'nan']
print(f"Missing Postal Codes: {len(missing_postal)}")
if len(missing_postal) > 0:
    print(missing_postal[['City', 'State', 'Country']].drop_duplicates())

In [ ]:
data['Postal Code'] = data['Postal Code'].replace('nan', '05401')
print(" Missing Postal Codes filled with '05401'")
print(f"Remaining nulls in Postal Code: {data['Postal Code'].isna().sum()}")

### 5.2 - Duplicate Rows

In [ ]:
dup_count = data.duplicated().sum()
print(f"Duplicate rows found: {dup_count}")

In [ ]:
if dup_count > 0:
    print("  Removing duplicate rows...")
    data.drop_duplicates(inplace=True)
    data.reset_index(drop=True, inplace=True)
    print(f" Duplicates removed. New shape: {data.shape}")
else:
    print(" No duplicate rows found.")

### 5.3 - Unwanted Whitespaces
Check all object (string) columns for leading/trailing whitespace.

In [ ]:
whitespace_report = {}
for col in data.select_dtypes(include='object').columns:
    has_ws = (data[col] != data[col].str.strip()).sum()
    if has_ws > 0:
        whitespace_report[col] = has_ws

if whitespace_report:
    print("  Whitespace issues found:")
    for col, count in whitespace_report.items():
        print(f"   - {col}: {count} values with extra whitespace")
else:
    print(" No leading/trailing whitespace issues found.")

In [ ]:
# Clean whitespace from all object columns
for col in data.select_dtypes(include='object').columns:
    data[col] = data[col].str.strip()

print(" All string columns stripped of leading/trailing whitespace.")

### 5.4 - Spelling & Consistency Checks
Inspect unique values of key categorical columns for typos, inconsistent casing, or unexpected entries.

In [ ]:
cat_cols = ['Ship Mode', 'Segment', 'Country', 'Region', 'Category', 'Sub-Category']

for col in cat_cols:
    unique_vals = data[col].unique()
    print(f"\n {col} ({len(unique_vals)} unique):")
    for v in sorted(unique_vals):
        print(f"   - {v}")

In [ ]:
# Standardize casing to Title Case for consistency
for col in cat_cols:
    data[col] = data[col].str.title()

print(" Categorical columns standardized to Title Case.")

In [ ]:
# Verify after cleaning
for col in cat_cols:
    print(f"{col}: {data[col].nunique()} unique -> {sorted(data[col].unique())}")

### 5.5 - Outlier Detection (IQR Method)

In [ ]:
num_cols = ['Sales', 'Days to Ship']

fig, axes = plt.subplots(1, len(num_cols), figsize=(14, 5))
for i, col in enumerate(num_cols):
    sns.boxplot(y=data[col], ax=axes[i], color='#5dade2')
    axes[i].set_title(f'Boxplot - {col}')
plt.suptitle('Outlier Detection via Boxplots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return outliers, lower, upper

for col in num_cols:
    outliers, lower, upper = detect_outliers_iqr(data, col)
    print(f"\n {col}:")
    print(f"   IQR bounds: [{lower:.2f}, {upper:.2f}]")
    print(f"   Outliers  : {len(outliers)} ({len(outliers)/len(data)*100:.2f}%)")

### 5.5.1 - Handle Sales Outliers - Capping (Winsorization)
Instead of removing high-value sales (they can be legitimate large orders), we **cap** them at the upper IQR fence.

In [ ]:
Q1 = data['Sales'].quantile(0.25)
Q3 = data['Sales'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

before = data['Sales'].describe()
data['Sales'] = data['Sales'].clip(upper=upper_fence)
after = data['Sales'].describe()

pd.DataFrame({'Before': before, 'After': after})

### 5.6 - Data Quality Summary After Cleaning

In [ ]:
print("=" * 50)
print("    DATA QUALITY SUMMARY AFTER CLEANING")
print("=" * 50)
print(f"  Shape          : {data.shape}")
print(f"  Nulls          : {data.isnull().sum().sum()}")
print(f"  Duplicates     : {data.duplicated().sum()}")
print(f"  Date cols      : Order Date ({data['Order Date'].dtype}), Ship Date ({data['Ship Date'].dtype})")
print(f"  Postal Code    : {data['Postal Code'].dtype}")
print("=" * 50)

---
## 6 - Distribution Analysis 

### 6.1 - Numerical Distributions

In [ ]:
fig = px.histogram(data, x='Sales', nbins=50, marginal='box',
                   color_discrete_sequence=['#2ecc71'],
                   title='Distribution of Sales')
fig.show()

In [ ]:
fig = px.histogram(data, x='Days to Ship', nbins=20, marginal='box',
                   color_discrete_sequence=['#3498db'],
                   title='Distribution of Days to Ship')
fig.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data['Sales'], kde=True, ax=axes[0], color='#2ecc71')
axes[0].set_title('Sales Distribution (with KDE)')

sns.histplot(data['Days to Ship'], kde=True, ax=axes[1], color='#3498db', bins=15)
axes[1].set_title('Days to Ship Distribution (with KDE)')

plt.tight_layout()
plt.show()

### 6.2 - Categorical Distributions

In [ ]:
fig = px.bar(data['Category'].value_counts().reset_index(),
             x='Category', y='count',
             color='Category',
             title='Orders by Category',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

In [ ]:
fig = px.bar(data['Sub-Category'].value_counts().reset_index(),
             x='Sub-Category', y='count',
             color='Sub-Category',
             title='Orders by Sub-Category',
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
fig = px.pie(data, names='Segment', title='Customer Segment Distribution',
             color_discrete_sequence=px.colors.qualitative.Set3,
             hole=0.4)
fig.show()

In [ ]:
fig = px.pie(data, names='Region', title='Orders by Region',
             color_discrete_sequence=px.colors.qualitative.Pastel2,
             hole=0.4)
fig.show()

In [ ]:
fig = px.bar(data['Ship Mode'].value_counts().reset_index(),
             x='Ship Mode', y='count',
             color='Ship Mode',
             title='Shipping Mode Distribution',
             color_discrete_sequence=px.colors.qualitative.Bold)
fig.show()

In [ ]:
fig = px.histogram(data, x='Sales', color='Category', marginal='box',
                   barmode='overlay', opacity=0.7,
                   title='Sales Distribution by Category',
                   color_discrete_sequence=px.colors.qualitative.Set2)
fig.show()

### 6.3 - Top 10 States by Number of Orders

In [ ]:
top_states = data['State'].value_counts().head(10).reset_index()
fig = px.bar(top_states, x='State', y='count',
             title='Top 10 States by Order Count',
             color='count',
             color_continuous_scale='Teal')
fig.show()

---
## 7 - Final Cleaned Data Preview

In [ ]:
data.info()

In [ ]:
data.head(10)

In [ ]:
print(f"\n Final dataset shape: {data.shape}")
print(f"   Ready for further analysis & modeling!")